# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- Use Pandas to load, inspect, and clean the dataset appropriately. 
- Transform relevant columns to create measures that address the problem at hand.
- **conduct EDA: visualization and statistical measures to understand the structure of the data**
- **recommend a set of manufacturers to consider as well as specific airplanes conforming to the client's request**
- **discuss the relationship between serious injuries/airplane damage incurred and at least *two* factors at play in the incident. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.**

In [1]:
# loading relevant packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Exploratory Data Analysis  
- Load in the cleaned data

In [2]:
df = pd.read_csv('data/aviation_cleaned.csv', low_memory=False)
print(df.shape)
df.head()

(63746, 23)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Injury.Severity,Aircraft.damage,Make,Model,Amateur.Built,Number.of.Engines,...,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Total.OnBoard,Injury.Rate,Is.Destroyed,Make.Model,Year
0,20001214X42095,Accident,SEA83LA036,1983-01-01,Non-Fatal,Substantial,Cessna,182P,No,1.0,...,0.0,1.0,3.0,VMC,Approach,4.0,0.0,0.0,Cessna 182P,1983
1,20001214X42067,Accident,MKC83LA056,1983-01-01,Non-Fatal,Substantial,Cessna,182RG,No,1.0,...,0.0,0.0,2.0,VMC,Landing,2.0,0.0,0.0,Cessna 182RG,1983
2,20001214X42063,Accident,MKC83LA050,1983-01-01,Non-Fatal,Substantial,Cessna,182P,No,1.0,...,0.0,0.0,1.0,VMC,Takeoff,1.0,0.0,0.0,Cessna 182P,1983
3,20001214X42018,Accident,LAX83FUG11,1983-01-01,Non-Fatal,Substantial,Piper,PA-28R-200,No,1.0,...,0.0,2.0,0.0,VMC,Approach,2.0,0.0,0.0,Piper PA-28R-200,1983
4,20001214X41951,Accident,CHI83LA074,1983-01-01,Non-Fatal,Substantial,Cessna,140,No,1.0,...,0.0,0.0,2.0,VMC,Landing,2.0,0.0,0.0,Cessna 140,1983


## Explore safety metrics across models/makes
- Remember that the client is interested in separate recommendations for smaller airplanes and larger airplanes. Choose a passenger threshold of 20 and separate the plane types. 

In [5]:
# Splitting into small and large aircraft based on total people on board
# Threshold of 20 passengers for small vs large
small = df[df['Total.OnBoard'] <= 20].copy()
large = df[df['Total.OnBoard'] > 20].copy()

print(f"Small aircraft accidents: {len(small)}")
print(f"Large aircraft accidents: {len(large)}")
print(f"Small aircraft mean Injury Rate: {small['Injury.Rate'].mean()}")
print(f"Large aircraft mean Injury Rate: {large['Injury.Rate'].mean()}")

Small aircraft accidents: 62539
Large aircraft accidents: 1207
Small aircraft mean Injury Rate: 0.27849194273265526
Large aircraft mean Injury Rate: 0.12104948387684872


- computing safety statistics for large aircraft makes with a minimum threshold of of 10 accidents to make statistically reliable calculations
- I'm using a lower threshold here as compared to the small aircrafts make since large aircraft accidents are rarer than small ones

In [12]:
large_makes = (large.groupby('Make')
                         .agg(
                             mean_injury_rate=('Injury.Rate', 'mean'),
                             mean_destroyed=('Is.Destroyed', 'mean'),
                             n_accidents=('Injury.Rate', 'count')
                         )
                         .reset_index()
                         .query('n_accidents >= 10')
                         .sort_values('mean_injury_rate'))

print(f"Large aircraft makes with >= 10 accidents: {len(large_makes)}")
large_makes.head(10)

Large aircraft makes with >= 10 accidents: 9


,Make,mean_injury_rate,mean_destroyed,n_accidents
1,Aerospatiale,0.066336,0.055556,18
15,Mcdonnell Douglas,0.083765,0.129032,155
3,Airbus Industrie,0.108301,0.130435,69
10,Embraer,0.111087,0.097222,72
6,Boeing,0.114345,0.116643,703
2,Airbus,0.130589,0.133333,75
14,Lockheed,0.175101,0.250000,20
8,De Havilland,0.190055,0.217391,23
9,Douglas,0.219753,0.261905,42


In [13]:
print("Large aircraft makes with >= 50 accidents:", 
      len(large_makes[large_makes['n_accidents'] >= 50]))
print("Large aircraft makes with >= 20 accidents:", 
      len(large_makes[large_makes['n_accidents'] >= 20]))
print("Large aircraft makes with >= 10 accidents:", 
      len(large_makes[large_makes['n_accidents'] >= 10]))

Large aircraft makes with >= 50 accidents: 5
Large aircraft makes with >= 20 accidents: 8
Large aircraft makes with >= 10 accidents: 9


Small aircrafts makes statitistics with a threshold of 20 passengers and a minimum of 50 accidents per make to ensure reliable statistics

In [11]:
small_makes = (small.groupby('Make')
                         .agg(
                             mean_injury_rate=('Injury.Rate', 'mean'),
                             mean_destroyed=('Is.Destroyed', 'mean'),
                             n_accidents=('Injury.Rate', 'count')
                         )
                         .reset_index()
                         .query('n_accidents >= 50')
                         .sort_values('mean_injury_rate'))

print(f"Small aircraft makes with >= 50 accidents: {len(small_makes)}")
small_makes.head(15)

Small aircraft makes with >= 50 accidents: 77


,Make,mean_injury_rate,mean_destroyed,n_accidents
76,Waco,0.103406,0.087591,137
42,Grumman-Schweizer,0.111549,0.251969,127
44,Helio,0.150144,0.123810,105
49,Let,0.153061,0.102041,98
52,Maule,0.154159,0.093146,569
11,Aviat Aircraft Inc,0.162338,0.038961,77
37,Great Lakes,0.163793,0.137931,58
45,Hiller,0.166667,0.197133,279
30,Enstrom,0.167472,0.159420,207
18,Boeing Stearman,0.180000,0.100000,50


#### Analyzing Makes

Explore the human injury risk profile for small and larger Makes:
- choose the 15 makes for each group possessing the lowest mean fatal/seriously injured fraction
- plot the mean fatal/seriously injured fraction for each of these subgroups side-by-side

**Distribution of injury rates: small makes**

Use a violinplot to look at the distribution of the fraction of passengers serious/fatally injured for small airplane makes. Just display makes with the ten lowest mean serious/fatal injury rates.

**Distribution of injury rates: large makes**

Use a stripplot to look at the distribution of the fraction of passengers serious/fatally injured for large airplane makes. Just display makes with the ten lowest mean serious/fatal injury rates.

**Evaluate the rate of aircraft destruction for both small and large aircraft by Make.** 

Sort your results and keep the lowest 15.

#### Provide a short discussion on your findings for your summary statistics and plots:
- Make any recommendations for Makes here based off of the destroyed fraction and fraction fatally/seriously injured
- Comment on the calculated statistics and any corresponding distributions you have visualized.

### Analyze plane types
- plot the mean fatal/seriously injured fraction for both small and larger planes 
- also provide a distributional plot of your choice for the fatal/seriously injured fraction by airplane type (stripplot, violin, etc)  
- filter ensuring that you have at least ten individual examples in each model/make to average over

**Larger planes**

**Smaller planes**
- for smaller planes, limit your plotted results to the makes with the 10 lowest mean serious/fatal injury fractions

### Discussion of Specific Airplane Types
- Discuss what you have found above regarding passenger fraction seriously/ both small and large airplane models.

### Exploring Other Variables
- Investigate how other variables effect aircraft damage and injury. You must choose **two** factors out of the following but are free to analyze more:

- Weather Condition
- Engine Type
- Number of Engines
- Phase of Flight
- Purpose of Flight

For each factor provide a discussion explaining your analysis with appropriate visualization / data summaries and interpreting your findings.